In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [2]:
df =pd.read_csv('train.txt',sep = ";",header=None ,names=['text','emotion'])

In [3]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull()

,text,emotion
0,False,False
1,False,False
2,False,False
3,False,False
4,False,False
...,...,...
15995,False,False
15996,False,False
15997,False,False
15998,False,False


In [5]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [6]:
df['emotion'].unique()

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [7]:
unique_emontions=df['emotion'].unique()
emotion_numbers={}
i=0
for emo in unique_emontions:
  emotion_numbers[emo] = i 
  i+=1

df['emotion']=df['emotion'].map(emotion_numbers)
print(emotion_numbers)

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}


In [8]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


# Text 

In [9]:
# lower case
df['text'] = df['text'].apply(lambda x:x.lower())

In [10]:
#  Remove Punctuation

import string 

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [11]:
df['text']=df['text'].apply(remove_punc)

In [12]:
# remove numbers 

def remove_numbers(txt):
  new=""
  for i in txt :
    if not i.isdigit():
      new=new+i 
  return new 

df['text']=df['text'].apply(remove_numbers)

In [13]:
# remove emoji or special characters 

def remove_emojis(txt):
  new=""
  for i in txt :
    if i.isascii():
      new+=i 
  return new 

df['text']=df['text'].apply(remove_emojis)

In [14]:
#  Remove stopwords 

import nltk

In [15]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [16]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sahil\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sahil\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [17]:
stop_words=set(stopwords.words('english'))

len(stop_words)

198

In [18]:
#  Tokenization 

def remove(txt):
  words=word_tokenize(txt)
  cleaned=[]
  for i in words :
    if not i in stop_words:
      cleaned.append(i)
  
  return ' '.join(cleaned)


In [19]:
df['text']=df['text'].apply(remove)

In [20]:
df['text']

0                                    didnt feel humiliated
1        go feeling hopeless damned hopeful around some...
2                im grabbing minute post feel greedy wrong
3        ever feeling nostalgic fireplace know still pr...
4                                          feeling grouchy
                               ...                        
15995        brief time beanbag said anna feel like beaten
15996    turning feel pathetic still waiting tables sub...
15997                             feel strong good overall
15998                       feel like rude comment im glad
15999                         know lot feel stupid portray
Name: text, Length: 16000, dtype: object

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'],df['emotion'] , test_size=0.20, random_state=42)

In [22]:
X_train
X_test

8756                             ive made week feel beaten
4660                              feel strategy worthwhile
6095                     feel worthless weak say want find
304                                        feel clever nov
8241                      im moved ive feeling kind gloomy
                               ...                        
15578    feel useful pulpit find ironic often question ...
5746             dried bladders ready day im feeling brave
6395                             feel thrilled matter days
7624     woke morning text mr c declaring walking work ...
15245                                            feel dumb
Name: text, Length: 3200, dtype: object

In [23]:
y_train

676      5
12113    0
7077     2
13005    0
12123    1
        ..
13418    4
5390     2
860      0
15795    0
7270     1
Name: emotion, Length: 12800, dtype: int64

In [24]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Bag of Words vectorization
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)   # fit on train
X_test_bow = bow_vectorizer.transform(X_test)         # only transform test

# Train Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

# Predictions
pred_bow = nb_model.predict(X_test_bow)

# Accuracy
print("Accuracy:", accuracy_score(y_test, pred_bow))


Accuracy: 0.7678125


In [25]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0], dtype=int64)

In [26]:
y_test

8756     0
4660     5
6095     0
304      5
8241     0
        ..
15578    5
5746     5
6395     5
7624     5
15245    0
Name: emotion, Length: 3200, dtype: int64

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
# TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer()

In [28]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)   # fit on train
X_test_tfidf = tfidf_vectorizer.transform(X_test)         # only transform test

# Train Naive Bayes
nb_model_2 = MultinomialNB()
nb_model_2.fit(X_train_tfidf, y_train)

# Predictions
pred_tfidf = nb_model_2.predict(X_test_tfidf)

# Accuracy
print("Accuracy:", accuracy_score(y_test, pred_tfidf))

Accuracy: 0.6609375


In [29]:
from sklearn.linear_model import LogisticRegression 

In [30]:
model_reg=LogisticRegression(max_iter=1000)

model_reg.fit(X_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [31]:
log_pred=model_reg.predict(X_test_tfidf)

In [32]:
print(accuracy_score(y_test,log_pred))

0.8615625


In [33]:
import joblib
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["label"] = le.fit_transform(df["emotion"])

joblib.dump(model_reg, 'regression_sentimentanalysis.pkl')
# joblib.dump(scaler,'scaler.pkl')
joblib.dump(tfidf_vectorizer, "scaler.pkl")
joblib.dump(le, "label_encoder.pkl")
joblib.dump(emotion_numbers, "emotion_numbers.pkl")

joblib.dump(df['text'].to_list(),'columns.pkl')

['columns.pkl']